# Adjoint AD, compiled to the GPU

**Record a valuation once, replay it twenty million times.**

This notebook is the Jupyter analogy of [`demo/greeks-on-gpu.sh`](../demo/greeks-on-gpu.sh):
a down-and-in put written as a plain-Java lambda, recorded once, then priced
with `price + delta + vega + rho + dV/dK + dV/dT` from **one reverse sweep** —
then a spot ladder and a crash scenario on the *same* compiled kernel, no rebuild.

Every call below is the real Java method: the `nablatensor` package just boots a
JDK 25 JVM through JPype and forwards to `com.nablatensor.quant.*`. See
[`notebooks/README.md`](README.md) for setup.

## 1 · The position

A desk holds a **down-and-in put**: strike 100, knock-in at 70. Today it is
worthless paper — but if the stock ever trades through 70 it springs to life
as a put and the desk is long a lot of downside.

What is it worth, and what hedges it?

In [ ]:
import nablatensor as nt

nt.start()  # boots one JDK 25 JVM against the checkout's */target/classes

for e in nt.engines():
    mark = "ok" if e["available"] else "--"
    print(f"  {mark:>3}  {e['name']:<10} {e['describe']}")

# Fastest fp32 adjoint backend this machine has: vulkan, else rocm, else cpu-jit.
# Override with e.g. ENGINE = "cpu-jit" for a pure-Java run.
ENGINE = nt.best_engine()
print(f"\nusing engine: {ENGINE}")

In [ ]:
def greek_row(label, value, note=""):
    """The notebook's stand-in for the shell demo's row() helper."""
    print(f"   {label:<9} {value:>16,.4f}   {note}")

## 2 · The market is a record. Every field is a risk factor.

Five doubles: spot, strike, vol, rate, maturity. No framework, no annotations,
no risk factor named with a string. `EquityMarket.vol` is checked by the
compiler and impossible to misspell.

In [ ]:
market = nt.EquityMarket(100.0, 100.0, 0.28, 0.03, 1.0)
print(market)

## 3 · The payoff is a lambda. Plain Java. No AD, no GPU.

A down-and-in put, monitored every step with a smoothed indicator so the whole
payoff stays differentiable. The barrier at 70 is just a number in the lambda —
but its sensitivity still falls out of the same sweep.

In [ ]:
note = nt.ExoticProducts.barrier(
    nt.OptionType.PUT, nt.ExoticProducts.Barrier.DOWN_IN, 70.0, 1.0)
note

## 4 · Record once. Compile once. Then never again.

`build()` records the tape and turns it into a kernel — tape → SPIR-V compute
shader → Vulkan. That is the slow part, and it happens exactly once. After this,
`run(...)` is just a launch.

In [ ]:
mc = (nt.MonteCarlo.of(note)
        .market(market)
        .steps(252)
        .fp32()
        .greeks()
        .on(ENGINE)
        .build())

print(f"   engine {mc.engine()}   {mc.nodes()} tape nodes")

## 5 · Price it. Twenty million paths. Every Greek at once.

One forward sweep for the price, one reverse sweep for **all five**
sensitivities. Bump-and-revalue needs a full re-run per input, and the count
only ever grows.

In [ ]:
p = mc.run(20_000_000, 42)

greek_row("price", p.price(),             "per 100 notional")
greek_row("delta", p.delta(),             "<-- shares to hedge with. THIS is the number.")
greek_row("vega",  p.vega(),              "long vol, as a down-and-in put is")
greek_row("rho",   p.rho(),               "rate sensitivity")
greek_row("dV/dK", p.strikeSensitivity(), "strike sensitivity")
greek_row("dV/dT", p.greeks().maturity(), "time decay")
print(f"   {p.scenarios():,} paths in {p.seconds():.2f} s  =  "
      f"{p.scenariosPerSecond():.2e} paths/s  on {p.engine()}")

## 6 · Move the market. Same kernel, no rebuild — it is an argument.

`mc.run(state, ...)` takes a fresh `EquityMarket` as a kernel argument, so
nothing recompiles. Delta runs from steeply negative near the barrier to
nearly flat far above it: close to 70 the option is alive and tracks the put;
up at 120 the knock-in rarely triggers. That curvature is the barrier risk —
exactly where bump-and-revalue returns noise instead of a number.

In [ ]:
print(f"   {'spot':>6} {'price':>12} {'delta':>12} {'vega':>12}")
for s in range(80, 121, 10):
    q = mc.run(market.withSpot(float(s)), 5_000_000, 42)
    print(f"   {s:6.0f} {q.price():12.4f} {q.delta():12.4f} {q.vega():12.3f}")

In [ ]:
# A finer grid, plotted. Optional — needs matplotlib.
try:
    import matplotlib.pyplot as plt

    spots = list(range(55, 141, 5))
    ladder = [mc.run(market.withSpot(float(s)), 2_000_000, 42) for s in spots]
    prices = [q.price() for q in ladder]
    deltas = [q.delta() for q in ladder]

    fig, (ax_p, ax_d) = plt.subplots(1, 2, figsize=(11, 4))
    ax_p.plot(spots, prices); ax_p.axvline(70, ls="--", c="0.6")
    ax_p.set_title("price"); ax_p.set_xlabel("spot")
    ax_d.plot(spots, deltas); ax_d.axvline(70, ls="--", c="0.6")
    ax_d.set_title("delta (one reverse sweep per point)"); ax_d.set_xlabel("spot")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping the plot")

## 7 · Now the market gaps down. Spot 72, vol 55%.

One gap and the note that was worthless paper this morning is deep in the
money with the knock-in near-certain — and delta lurches. The hedge moves with
it, from the same kernel.

In [ ]:
crashed = market.withSpot(72.0).withVol(0.55)
c = mc.run(crashed, 20_000_000, 42)

greek_row("price", c.price(), "worth ~6 a note this morning; knock-in now near-certain")
greek_row("delta", c.delta(), "one gap moved the hedge. Sell more stock.")
greek_row("vega",  c.vega())

In [ ]:
mc.close()  # free the device kernel; the JVM stays up for the rest of the kernel session

---

The quant wrote a payoff. The desk got a hedge that moves with it. Neither of
them typed the word *Vulkan*.

See also [`demo/adjoint-vs-bump.sh`](../demo/adjoint-vs-bump.sh) — the same five
Greeks computed two ways, adjoint versus central bump-and-revalue.